# wafrha tenwarha — Smart Electricity Consumption Analyzer


In [ ]:
!pip install ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, silhouette_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

In [ ]:
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=235)
raw = dataset.data.features

print(raw.shape)
raw.head()

In [ ]:
df = raw[['Date', 'Time', 'Global_active_power']].copy()
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
df['power_kw'] = pd.to_numeric(df['Global_active_power'], errors='coerce')
df = df.set_index('datetime')

print('Missing minutes:', df['power_kw'].isna().sum())

In [ ]:
hourly = df['power_kw'].resample('h').mean()
hourly = hourly.interpolate(limit=3)

valid_hours = hourly.resample('D').count()
daily = pd.DataFrame({'kwh': hourly.resample('D').mean() * 24})
daily = daily[valid_hours >= 20]
daily = daily.asfreq('D').interpolate()

print('Days:', len(daily))
daily.describe()

In [ ]:
daily['avg_7'] = daily['kwh'].rolling(7).mean()
daily['avg_30'] = daily['kwh'].rolling(30).mean()

plt.plot(daily['kwh'], color='lightgray', label='Daily kWh')
plt.plot(daily['avg_7'], label='7-day average')
plt.plot(daily['avg_30'], label='30-day average', linewidth=2)
plt.title('Daily Electricity Consumption')
plt.ylabel('kWh')
plt.legend()
plt.show()

In [ ]:
for days in [7, 30]:
    recent = daily['kwh'].iloc[-days:].mean()
    before = daily['kwh'].iloc[-2 * days:-days].mean()
    change = (recent - before) / before * 100
    trend = 'increased' if change > 0 else 'decreased'
    print(f'Last {days} days: {recent:.1f} kWh/day, {trend} by {abs(change):.1f}% vs the previous {days} days')

In [ ]:
hourly_df = hourly.dropna().to_frame('kw')
hourly_df['hour'] = hourly_df.index.hour
hourly_df['dayofweek'] = hourly_df.index.dayofweek

day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

hour_avg = hourly_df.groupby('hour')['kw'].mean()
hour_avg.plot(kind='bar', color='steelblue')
plt.title('Average Power by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('kW')
plt.show()

print('Top 3 peak hours:', hour_avg.nlargest(3).index.tolist())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

dow_avg = daily['kwh'].groupby(daily.index.dayofweek).mean()
dow_avg.index = day_names
dow_avg.plot(kind='bar', ax=axes[0], color='darkorange')
axes[0].set_title('Average Daily kWh by Day of Week')
axes[0].set_ylabel('kWh')

month_avg = daily['kwh'].groupby(daily.index.month).mean()
month_avg.plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title('Average Daily kWh by Month')
axes[1].set_xlabel('Month')

plt.show()

In [ ]:
heat = hourly_df.pivot_table(index='dayofweek', columns='hour', values='kw')
heat.index = day_names

plt.figure(figsize=(14, 4))
sns.heatmap(heat, cmap='YlOrRd')
plt.title('Average Power (kW): Day of Week vs Hour')
plt.show()

In [ ]:
features = ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'avg_7', 'dayofweek', 'month']

data = pd.DataFrame({'kwh': daily['kwh']})
for lag in [1, 2, 3, 7]:
    data[f'lag_{lag}'] = data['kwh'].shift(lag)
data['avg_7'] = data['kwh'].shift(1).rolling(7).mean()
data['dayofweek'] = data.index.dayofweek
data['month'] = data.index.month
data = data.dropna()

split = int(len(data) * 0.8)
train = data.iloc[:split]
test = data.iloc[split:]

print('Train days:', len(train), '| Test days:', len(test))

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
}


def score(actual, predicted):
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted),
    }


results = {'Yesterday baseline': score(test['kwh'], test['lag_1'])}
predictions = {}

for name, model in models.items():
    model.fit(train[features], train['kwh'])
    predictions[name] = model.predict(test[features])
    results[name] = score(test['kwh'], predictions[name])

results = pd.DataFrame(results).T.round(3)
results

In [ ]:
best_name = results.drop('Yesterday baseline')['RMSE'].idxmin()
print('Best model:', best_name)

last = 90
plt.plot(test.index[-last:], test['kwh'].iloc[-last:], label='Actual', color='black')
for name in models:
    plt.plot(test.index[-last:], predictions[name][-last:], label=name)
plt.title(f'Actual vs Predicted (last {last} test days)')
plt.ylabel('kWh')
plt.legend()
plt.show()

In [ ]:
def make_row(history, date):
    return pd.DataFrame({
        'lag_1': [history.iloc[-1]],
        'lag_2': [history.iloc[-2]],
        'lag_3': [history.iloc[-3]],
        'lag_7': [history.iloc[-7]],
        'avg_7': [history.iloc[-7:].mean()],
        'dayofweek': [date.dayofweek],
        'month': [date.month],
    })


def forecast_next_days(model, kwh, days):
    history = kwh.copy()
    dates, values = [], []
    for _ in range(days):
        next_date = history.index[-1] + pd.Timedelta(days=1)
        value = model.predict(make_row(history, next_date)[features])[0]
        history.loc[next_date] = value
        dates.append(next_date)
        values.append(value)
    return pd.Series(values, index=dates, name='forecast_kwh')


best_model = models[best_name]
best_model.fit(data[features], data['kwh'])

next_week = forecast_next_days(best_model, daily['kwh'], 7)
print(f'Tomorrow ({next_week.index[0].date()}): {next_week.iloc[0]:.1f} kWh')
next_week.round(1)

In [ ]:
plt.plot(daily['kwh'].iloc[-30:], label='Last 30 days')
plt.plot(next_week, marker='o', color='red', label='Forecast')
plt.title('7-Day Forecast')
plt.ylabel('kWh')
plt.legend()
plt.show()

In [ ]:
previous = daily['kwh'].shift(1)
daily['mean_30'] = previous.rolling(30).mean()
daily['std_30'] = previous.rolling(30).std()
daily['limit'] = daily['mean_30'] + 2 * daily['std_30']
daily['is_anomaly'] = daily['kwh'] > daily['limit']

anomalies = daily[daily['is_anomaly']].copy()
anomalies['pct_above_avg'] = (anomalies['kwh'] / anomalies['mean_30'] - 1) * 100

print(f'Anomalous days: {len(anomalies)} ({len(anomalies) / len(daily) * 100:.1f}% of all days)')
anomalies[['kwh', 'mean_30', 'pct_above_avg']].sort_values('pct_above_avg', ascending=False).head(10).round(1)

In [ ]:
plt.plot(daily['kwh'], color='lightgray', label='Daily kWh')
plt.plot(daily['limit'], color='orange', linewidth=1, label='Anomaly limit')
plt.scatter(anomalies.index, anomalies['kwh'], color='red', s=15, label='Anomaly')
plt.title('Anomaly Detection')
plt.ylabel('kWh')
plt.legend()
plt.show()

In [ ]:
low_limit, high_limit = train['kwh'].quantile([0.33, 0.66])
bins = [-np.inf, low_limit, high_limit, np.inf]
labels = ['Low', 'Medium', 'High']

y_train = pd.cut(train['kwh'], bins=bins, labels=labels)
y_test = pd.cut(test['kwh'], bins=bins, labels=labels)

print(f'Low: below {low_limit:.1f} kWh | Medium: up to {high_limit:.1f} kWh | High: above {high_limit:.1f} kWh')

clf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
clf.fit(train[features], y_train)
y_pred = clf.predict(test[features])

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

pd.Series(clf.feature_importances_, index=features).sort_values().plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Feature Importance')

plt.show()

In [ ]:
daily['level'] = pd.cut(daily['kwh'], bins=bins, labels=labels)
print(daily['level'].value_counts())

tomorrow = daily.index[-1] + pd.Timedelta(days=1)
tomorrow_level = clf.predict(make_row(daily['kwh'], tomorrow)[features])[0]
print(f'Predicted level for {tomorrow.date()}: {tomorrow_level}')

In [ ]:
hourly_df['date'] = hourly_df.index.normalize()
profile = hourly_df.pivot_table(index='date', columns='hour', values='kw').dropna()
hour_cols = list(profile.columns)

print('Days used for clustering:', len(profile))

In [ ]:
inertias, silhouettes = [], []
k_values = range(2, 8)

for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(profile[hour_cols])
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(profile[hour_cols], km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(k_values, inertias, marker='o')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of clusters')
axes[0].set_ylabel('Inertia')
axes[1].plot(k_values, silhouettes, marker='o', color='green')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of clusters')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
profile['cluster'] = kmeans.fit_predict(profile[hour_cols])

centers = profile.groupby('cluster')[hour_cols].mean()
centers.T.plot(linewidth=2)
plt.title('Average Daily Pattern of Each Cluster')
plt.xlabel('Hour')
plt.ylabel('kW')
plt.legend(title='Cluster')
plt.show()

In [ ]:
profile['daily_kwh'] = profile[hour_cols].mean(axis=1) * 24
profile['is_weekend'] = profile.index.dayofweek >= 5

summary = profile.groupby('cluster').agg(
    days=('daily_kwh', 'size'),
    avg_kwh=('daily_kwh', 'mean'),
    weekend_share=('is_weekend', 'mean'),
)
summary['weekend_share'] = (summary['weekend_share'] * 100).round(1)
summary.round(1)

In [ ]:
hourly.rename('kw').rename_axis('datetime').to_csv('hourly_power.csv')

joblib.dump(best_model, 'forecast_model.joblib')
joblib.dump(clf, 'level_classifier.joblib')
joblib.dump(kmeans, 'kmeans_model.joblib')

print('Saved: hourly_power.csv, forecast_model.joblib, level_classifier.joblib, kmeans_model.joblib')